1. 라이브러리 import
2. 데이터 다시 불러오기
3. 사용할 컬럼 정의
4. X / y 분리
5. 범주형 / 수치형 컬럼 정의
6. 전처리기 구성
   ├─ 범주형 → OneHotEncoder
   └─ 수치형 → StandardScaler
7. Train에서 fit
8. Train / Validation / Test transform
9. 변환 결과 shape 확인
10. 전처리 결과 확인

In [2]:
import pandas as pd

train_df = pd.read_csv("../data/long_basic/train.csv")
val_df = pd.read_csv("../data/long_basic/validation.csv")
test_df = pd.read_csv("../data/long_basic/test.csv")

In [3]:
categorical_cols = [
    "league",
    "position_group"
]

numeric_cols = [
    "age",
    "starts",
    "minutes",
    "goals",
    "assists",
    "non_penalty_goals",
    "penalty_goals",
    "penalty_attempts",
    "goals_per90",
    "assists_per90",
    "goal_contrib_per90"
]

feature_cols = categorical_cols + numeric_cols

X_train = train_df[feature_cols]
y_train = train_df["next_goals"]

X_val = val_df[feature_cols]
y_val = val_df["next_goals"]

X_test = test_df[feature_cols]
y_test = test_df["next_goals"]

In [4]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

In [5]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numeric_cols
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_cols
        )
    ]
)

In [6]:
X_train_processed = preprocessor.fit_transform(X_train)

In [7]:
X_val_processed = preprocessor.transform(X_val)
X_test_processed = preprocessor.transform(X_test)

In [8]:
print("X_train:", X_train.shape)
print("X_train_processed:", X_train_processed.shape)

print("X_val_processed:", X_val_processed.shape)
print("X_test_processed:", X_test_processed.shape)

X_train: (22430, 13)
X_train_processed: (22430, 18)
X_val_processed: (923, 18)
X_test_processed: (926, 18)


In [9]:
feature_names = preprocessor.get_feature_names_out()

feature_names

array(['num__age', 'num__starts', 'num__minutes', 'num__goals',
       'num__assists', 'num__non_penalty_goals', 'num__penalty_goals',
       'num__penalty_attempts', 'num__goals_per90', 'num__assists_per90',
       'num__goal_contrib_per90', 'cat__league_Bundesliga',
       'cat__league_La Liga', 'cat__league_Ligue 1',
       'cat__league_Premier League', 'cat__league_Serie A',
       'cat__position_group_FW', 'cat__position_group_MF'], dtype=object)

In [10]:
X_train_processed_df = pd.DataFrame(
    X_train_processed,
    columns=feature_names,
    index=X_train.index
)

X_train_processed_df.head()

,num__age,num__starts,num__minutes,num__goals,num__assists,num__non_penalty_goals,num__penalty_goals,num__penalty_attempts,num__goals_per90,num__assists_per90,num__goal_contrib_per90,cat__league_Bundesliga,cat__league_La Liga,cat__league_Ligue 1,cat__league_Premier League,cat__league_Serie A,cat__position_group_FW,cat__position_group_MF
0,1.350174,0.514820,0.497433,0.228668,-1.005878,0.347960,-0.352813,-0.383039,0.110929,-1.094686,-0.395353,0.0,0.0,1.0,0.0,0.0,1.0,0.0
1,0.054895,-0.921004,-0.720548,-0.207325,-0.605434,-0.142565,-0.352813,-0.383039,0.016236,-0.513519,-0.213188,1.0,0.0,0.0,0.0,0.0,1.0,0.0
2,-0.722273,-1.573651,-1.399294,0.010671,0.195456,-0.142565,0.608975,1.252826,0.886191,1.362363,1.285300,0.0,0.0,0.0,0.0,1.0,1.0,0.0
3,-0.204161,1.428526,1.356447,1.100655,-0.204989,1.329012,-0.352813,-0.383039,0.598593,-0.479260,0.252906,0.0,0.0,0.0,1.0,0.0,1.0,0.0
4,0.313951,-0.137827,-0.156233,1.318652,-1.005878,1.329012,0.608975,0.434894,1.683900,-1.094686,0.822906,1.0,0.0,0.0,0.0,0.0,1.0,0.0


In [11]:
X_train_processed_df[
    [col for col in feature_names if col.startswith("num__")]
].mean()

num__age                  -2.116106e-16
num__starts               -9.883608e-17
num__minutes               2.534259e-18
num__goals                 4.308240e-17
num__assists              -5.068517e-18
num__non_penalty_goals     5.955508e-17
num__penalty_goals         7.602776e-18
num__penalty_attempts      4.054814e-17
num__goals_per90           5.068517e-18
num__assists_per90        -2.889055e-16
num__goal_contrib_per90    9.376757e-17
dtype: float64

In [12]:
X_train_processed_df[
    [col for col in feature_names if col.startswith("num__")]
].std()

num__age                   1.000022
num__starts                1.000022
num__minutes               1.000022
num__goals                 1.000022
num__assists               1.000022
num__non_penalty_goals     1.000022
num__penalty_goals         1.000022
num__penalty_attempts      1.000022
num__goals_per90           1.000022
num__assists_per90         1.000022
num__goal_contrib_per90    1.000022
dtype: float64

### 전처리 결과 확인

- 수치형 피처의 평균이 0에 매우 가깝게 변환되었다.
- 수치형 피처의 표준편차가 약 1로 변환되었다.
- 범주형 피처는 One-Hot Encoding을 통해 숫자형 컬럼으로 변환되었다.
- 최종 입력 피처 수는 13개에서 18개로 증가하였다.
- Train 기준으로 전처리기를 학습하고 Validation/Test에는 transform만 적용하여 데이터 누수를 방지하였다.